In [ ]:
# File: large-fluffy.ipynb
# Code: Claude Code and Codex
# Review: Ryoichi Ando (ryoichi.ando@zozo.com)
# License: Apache v2.0

In [ ]:
import os
import zipfile

import numpy as np
from frontend import App, get_cache_dir

# create an app
app = App.create("large-fluffy")

# fetch the volumetric asset on first use and cache it locally. It ships as a
# single archive, so expand it once and reuse it on later runs.
asset_root = os.path.join(get_cache_dir(), "tet-assets")
app.extra.sparse_clone(
    "https://github.com/wiso-enoji/Barrier-Free-Supplementary",
    asset_root,
    ["assets.zip"],
)
asset_dir = os.path.join(asset_root, "assets")
if not os.path.isdir(asset_dir):
    with zipfile.ZipFile(os.path.join(asset_root, "assets.zip")) as archive:
        archive.extractall(asset_root)

# load one soft ball (450k tetrahedra) and bring the asset down to a ball a
# little over half a unit across
ball_radius = 0.54
V, F, T = app.mesh.load_tet(os.path.join(asset_dir, "fluffy_ball.mesh"))
V *= ball_radius / np.abs(V).max()
app.asset.add.tet("ball", V, F, T)

# The press is a closed cylinder, generated here rather than loaded, so its
# resolution stays a knob of this example. It is deliberately COARSE: one ring
# of quads around, capped at both ends. A collider this much coarser than the
# balls it holds is the point of the example, because each collider vertex then
# couples to a large number of ball vertices and a single contact-matrix row
# grows to tens of thousands of columns. Raising n_side narrows those rows.
cyl_radius, cyl_height, n_side = 1.1415, 2.8293, 32
theta = 2.0 * np.pi * np.arange(n_side) / n_side
ring = np.stack(
    [cyl_radius * np.cos(theta), np.zeros(n_side), cyl_radius * np.sin(theta)],
    axis=1,
)
# the caps close the cylinder, so its own floor holds the balls up and the
# scene needs no separate ground. Each cap fans from a vertex at its center
# rather than from a point on its rim: a rim fan spans the full diameter with
# near-degenerate slivers, and crushing a soft body onto one of those is a test
# of the triangle rather than of the contact.
lo_hub, hi_hub = 2 * n_side, 2 * n_side + 1
cyl_V = np.concatenate(
    [
        ring,
        ring + [0, cyl_height, 0],
        [[0.0, 0.0, 0.0], [0.0, cyl_height, 0.0]],
    ]
)

i = np.arange(n_side)
j = (i + 1) % n_side
side = np.concatenate(
    [
        np.stack([i, n_side + i, j], -1),
        np.stack([j, n_side + i, n_side + j], -1),
    ]
)
cap_lo = np.stack([np.full_like(i, lo_hub), j, i], -1)
cap_hi = np.stack([np.full_like(i, hi_hub), n_side + i, n_side + j], -1)
cyl_F = np.concatenate([side, cap_lo, cap_hi]).astype(np.int32)
app.asset.add.tri("cylinder", cyl_V, cyl_F)

# create a scene
scene = app.scene.create()

# The cylinder: bottom anchored, top driven down as a piston. Every one of its
# vertices belongs to one of the two pinned rings, so it is fully kinematic and
# carries no elastic or bending energy of its own; setting a material on it
# would change nothing. It is marked invisible because the press is closed
# and would otherwise hide everything it holds; that is a drawing flag only,
# so it still collides with the balls exactly as before.
cyl = scene.add("cylinder").invisible()
cyl.pin(cyl.grab([0, -1, 0]))
piston = cyl.pin(cyl.grab([0, 1, 0]))

# Close the gap on an exponential schedule rather than a linear one: the press
# drops quickly at first and then creeps, so the balls spend most of the run in
# heavy contact instead of only reaching it at the end. The floor stops the
# piston just short of flat.
squeeze_end, gap_floor, n_step = 2.0, 0.05, 10
height = cyl_height
for k in range(n_step):
    target = max(cyl_height * 0.1 ** (2.0 * (k + 1) / n_step), gap_floor)
    piston.move_by(
        [0, target - height, 0],
        t_start=squeeze_end * k / n_step,
        t_end=squeeze_end * (k + 1) / n_step,
    )
    height = target

# then withdraw and let the stack recover
piston.move_by(
    [0, cyl_height, 0], t_start=squeeze_end, t_end=squeeze_end + 0.1
)

# stack the balls up the cylinder, offset from the axis and rotated a little
# each step so they interlock rather than sit in a perfect column
density, young_mod = 1e2, 1e4
n_ball, swirl, lateral = 5, 2.4, 0.55
rise = (cyl_height - 2.2 * ball_radius) / (n_ball - 1)
for k in range(n_ball):
    obj = scene.add("ball").at(
        lateral * np.cos(swirl * k),
        1.1 * ball_radius + k * rise,
        lateral * np.sin(swirl * k),
    )
    # Young's modulus is pre-normalized by density (the elastic energy density
    # is scaled by mass), so divide it through.
    (
        obj.param.set("density", density)
        .set("young-mod", young_mod / density)
        .set("poiss-rat", 0.4)
    )

# compile the scene and report stats
scene = scene.build().report()

# preview the initial scene
scene.preview()

In [ ]:
# create a new session with the compiled scene
session = app.session.create(scene)

# set session parameters
(
    session.param.set("auto-save", 10)
    .set("gravity", [0.0, -9.81, 0.0])
    .set("dt", 0.01)
    .set("fps", 100)
    .set("frames", 300)
    .set("target-toi", 0.999)
    .set("cg-tol", 0.0001)
    .set("csrmat-max-nnz", 100000000)
)

# build this session
session = session.build()

In [ ]:
# start the simulation (this example takes a long time)
session.start()

In [ ]:
# this example takes a long time...
# in case you shutdown the server (or kernel) and still want to restart
# from where you have (auto) saved, do this. Do not call cells above.

from frontend import App  # noqa

# recover the session from auto-saved state
session = App.recover("large-fluffy")

# resume if not currently running
if not App.busy():
    session.resume()

# preview the current state
session.preview()

# stream the logs
session.stream()